# Custom Solution - Allstate Claims Severity

В данном ноутбуке представлено кастомное решение для задачи регрессии (предсказание страховых выплат).

**Цель**: построить решение, превосходящее LAMA baseline (Log MAE = 0.4363).


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import DataLoader
from src.features.eda_insights import load_eda_insights
from src.models.custom import CustomPipeline, HyperparameterTuner, create_submission
from src.utils.logging_config import setup_logging, get_logger
from src.utils.metrics import compute_log_mae, compute_metrics

warnings.filterwarnings('ignore')
setup_logging()
logger = get_logger(__name__)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

# Результаты LAMA baseline из 02_lama_baseline.ipynb
LAMA_BASELINE_LOG_MAE = 0.4363
LAMA_BASELINE_RAW_MAE = 1187

## Загрузка данных


In [2]:
loader = DataLoader()
train_df = loader.load_train()
test_df = loader.load_test()

X = train_df.drop(columns=['id', 'loss'])
y = train_df['loss']
X_test = test_df.drop(columns=['id'])
test_ids = test_df['id']

cat_columns = loader.get_categorical_columns(train_df)
cont_columns = loader.get_continuous_columns(train_df)

eda_insights = load_eda_insights()

2025-12-21 18:05:40,699 - src.data.loader - INFO - Loaded 188,318 training samples
2025-12-21 18:05:41,066 - src.data.loader - INFO - Loaded 125,546 test samples


In [3]:
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

## Пайплайн 1: LightGBM

Градиентный бустинг на основе LightGBM:

- Логарифмирование таргета (рекомендация из EDA)
- Label encoding для категориальных признаков
- StandardScaler для непрерывных признаков
- 5-fold кросс-валидация

In [4]:
lgb_pipeline = CustomPipeline(
    model_type="lightgbm",
    n_folds=5,
    random_state=42,
    eda_insights=eda_insights,
    model_params={
        "n_estimators": 500,
        "num_leaves": 31,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    }
)

lgb_pipeline.fit(X, y, cat_columns, cont_columns)
log_mae_lgb, std_lgb = lgb_pipeline.get_cv_score(metric="log_mae")
raw_mae_lgb, _ = lgb_pipeline.get_cv_score(metric="raw_mae")

print(f"LightGBM CV Results:")
print(f"  Log MAE: {log_mae_lgb:.4f} (+/- {std_lgb:.4f})")
print(f"  Raw MAE: {raw_mae_lgb:.4f}")

lgb_pipeline.save_model(models_dir / "lightgbm_pipeline.joblib")

2025-12-21 18:05:41,155 - src.models.custom - INFO - Initialized CustomPipeline with model_type=lightgbm, n_folds=5
2025-12-21 18:05:41,155 - src.models.custom - INFO - Starting 5-fold CV with lightgbm
2025-12-21 18:05:41,155 - src.models.custom - INFO - Data shape: (188318, 130), Cat cols: 116, Cont cols: 14
2025-12-21 18:05:41,156 - src.models.custom - INFO - Target stats: mean=3037.34, std=2904.08, min=0.67, max=121012.25
2025-12-21 18:05:41,159 - src.models.custom - INFO - Applied log1p transform to target. New stats: mean=7.6859, std=0.8113
2025-12-21 18:05:41,216 - src.models.custom - INFO - Fold 1/5: Train=150,654, Val=37,664
2025-12-21 18:12:31,154 - src.models.custom - INFO - Fold 1/5: Log MAE = 0.4140, Raw MAE = 1135.22 (took 410.0s)
2025-12-21 18:12:31,228 - src.models.custom - INFO - Fold 2/5: Train=150,654, Val=37,664
2025-12-21 18:19:19,376 - src.models.custom - INFO - Fold 2/5: Log MAE = 0.4165, Raw MAE = 1154.67 (took 408.2s)
2025-12-21 18:19:19,439 - src.models.custom 

## Пайплайн 2: XGBoost

Градиентный бустинг на основе XGBoost:
- Аналогичный препроцессинг
- Регуляризация через max_depth


In [5]:
xgb_pipeline = CustomPipeline(
    model_type="xgboost",
    n_folds=5,
    random_state=42,
    eda_insights=eda_insights,
    model_params={
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    }
)

xgb_pipeline.fit(X, y, cat_columns, cont_columns)
log_mae_xgb, std_xgb = xgb_pipeline.get_cv_score(metric="log_mae")
raw_mae_xgb, _ = xgb_pipeline.get_cv_score(metric="raw_mae")

print(f"XGBoost CV Results:")
print(f"  Log MAE: {log_mae_xgb:.4f} (+/- {std_xgb:.4f})")
print(f"  Raw MAE: {raw_mae_xgb:.4f}")

xgb_pipeline.save_model(models_dir / "xgboost_pipeline.joblib")

2025-12-21 18:46:39,340 - src.models.custom - INFO - Initialized CustomPipeline with model_type=xgboost, n_folds=5
2025-12-21 18:46:39,340 - src.models.custom - INFO - Starting 5-fold CV with xgboost
2025-12-21 18:46:39,341 - src.models.custom - INFO - Data shape: (188318, 130), Cat cols: 116, Cont cols: 14
2025-12-21 18:46:39,341 - src.models.custom - INFO - Target stats: mean=3037.34, std=2904.08, min=0.67, max=121012.25
2025-12-21 18:46:39,343 - src.models.custom - INFO - Applied log1p transform to target. New stats: mean=7.6859, std=0.8113
2025-12-21 18:46:39,394 - src.models.custom - INFO - Fold 1/5: Train=150,654, Val=37,664
2025-12-21 18:53:30,634 - src.models.custom - INFO - Fold 1/5: Log MAE = 0.4140, Raw MAE = 1133.41 (took 411.3s)
2025-12-21 18:53:30,695 - src.models.custom - INFO - Fold 2/5: Train=150,654, Val=37,664
2025-12-21 19:00:24,148 - src.models.custom - INFO - Fold 2/5: Log MAE = 0.4174, Raw MAE = 1156.18 (took 413.5s)
2025-12-21 19:00:24,209 - src.models.custom - 

## Пайплайн 3: CatBoost

Градиентный бустинг на основе CatBoost:
- Эффективная обработка категориальных признаков
- Ordered boosting для уменьшения переобучения


In [6]:
catboost_pipeline = CustomPipeline(
    model_type="catboost",
    n_folds=5,
    random_state=42,
    eda_insights=eda_insights,
    model_params={
        "iterations": 500,
        "depth": 6,
        "learning_rate": 0.05,
    }
)

catboost_pipeline.fit(X, y, cat_columns, cont_columns)
log_mae_catboost, std_catboost = catboost_pipeline.get_cv_score(metric="log_mae")
raw_mae_catboost, _ = catboost_pipeline.get_cv_score(metric="raw_mae")

print(f"CatBoost CV Results:")
print(f"  Log MAE: {log_mae_catboost:.4f} (+/- {std_catboost:.4f})")
print(f"  Raw MAE: {raw_mae_catboost:.4f}")

catboost_pipeline.save_model(models_dir / "catboost_pipeline.joblib")

2025-12-21 19:27:55,735 - src.models.custom - INFO - Initialized CustomPipeline with model_type=catboost, n_folds=5
2025-12-21 19:27:55,735 - src.models.custom - INFO - Starting 5-fold CV with catboost
2025-12-21 19:27:55,735 - src.models.custom - INFO - Data shape: (188318, 130), Cat cols: 116, Cont cols: 14
2025-12-21 19:27:55,736 - src.models.custom - INFO - Target stats: mean=3037.34, std=2904.08, min=0.67, max=121012.25
2025-12-21 19:27:55,739 - src.models.custom - INFO - Applied log1p transform to target. New stats: mean=7.6859, std=0.8113
2025-12-21 19:27:55,791 - src.models.custom - INFO - Fold 1/5: Train=150,654, Val=37,664
2025-12-21 19:34:43,287 - src.models.custom - INFO - Fold 1/5: Log MAE = 0.4180, Raw MAE = 1145.58 (took 407.5s)
2025-12-21 19:34:43,350 - src.models.custom - INFO - Fold 2/5: Train=150,654, Val=37,664
2025-12-21 19:41:30,380 - src.models.custom - INFO - Fold 2/5: Log MAE = 0.4213, Raw MAE = 1166.38 (took 407.1s)
2025-12-21 19:41:30,443 - src.models.custom 

In [7]:
# Code for new session (to not retrain everything)

lgb_pipeline = CustomPipeline.load_model(models_dir / "lightgbm_pipeline.joblib")
log_mae_lgb, std_lgb = lgb_pipeline.get_cv_score(metric="log_mae")
raw_mae_lgb, _ = lgb_pipeline.get_cv_score(metric="raw_mae")

xgb_pipeline = CustomPipeline.load_model(models_dir / "xgboost_pipeline.joblib")
log_mae_xgb, std_xgb = xgb_pipeline.get_cv_score(metric="log_mae")
raw_mae_xgb, _ = xgb_pipeline.get_cv_score(metric="raw_mae")

catboost_pipeline = CustomPipeline.load_model(models_dir / "catboost_pipeline.joblib")
log_mae_catboost, std_catboost = catboost_pipeline.get_cv_score(metric="log_mae")
raw_mae_catboost, _ = catboost_pipeline.get_cv_score(metric="raw_mae")

print(f"LightGBM CV Results:")
print(f"  Log MAE: {log_mae_lgb:.4f} (+/- {std_lgb:.4f})")
print(f"  Raw MAE: {raw_mae_lgb:.4f}")

print(f"XGBoost CV Results:")
print(f"  Log MAE: {log_mae_xgb:.4f} (+/- {std_xgb:.4f})")
print(f"  Raw MAE: {raw_mae_xgb:.4f}")

print(f"CatBoost CV Results:")
print(f"  Log MAE: {log_mae_catboost:.4f} (+/- {std_catboost:.4f})")
print(f"  Raw MAE: {raw_mae_catboost:.4f}")

2025-12-21 20:08:43,360 - src.models.custom - INFO - Initialized CustomPipeline with model_type=lightgbm, n_folds=5
2025-12-21 20:08:43,361 - src.models.custom - INFO - Model loaded from ../models/lightgbm_pipeline.joblib
2025-12-21 20:08:43,361 - src.models.custom - INFO - Model type: lightgbm, CV Log MAE: 0.4155
2025-12-21 20:08:43,371 - src.models.custom - INFO - Initialized CustomPipeline with model_type=xgboost, n_folds=5
2025-12-21 20:08:43,371 - src.models.custom - INFO - Model loaded from ../models/xgboost_pipeline.joblib
2025-12-21 20:08:43,371 - src.models.custom - INFO - Model type: xgboost, CV Log MAE: 0.4158
2025-12-21 20:08:43,376 - src.models.custom - INFO - Initialized CustomPipeline with model_type=catboost, n_folds=5
2025-12-21 20:08:43,376 - src.models.custom - INFO - Model loaded from ../models/catboost_pipeline.joblib
2025-12-21 20:08:43,376 - src.models.custom - INFO - Model type: catboost, CV Log MAE: 0.4199
LightGBM CV Results:
  Log MAE: 0.4155 (+/- 0.0010)
  R

## Оптимизация гиперпараметров (Optuna)

Автоматический подбор гиперпараметров LightGBM с использованием Optuna (TPE сэмплер)

In [8]:
tuner = HyperparameterTuner(
    model_type="lightgbm",
    n_trials=5,
    timeout=30000,
    n_folds=5,
    random_state=42,
    eda_insights=eda_insights,
)

best_params = tuner.tune(X, y, cat_columns, cont_columns)
print(f"Best parameters: {best_params}")

2025-12-21 20:08:43,386 - src.models.custom - INFO - Starting hyperparameter tuning for lightgbm
2025-12-21 20:08:43,386 - src.models.custom - INFO - Tuning config: n_trials=5, timeout=30000s, n_folds=5
2025-12-21 20:08:43,386 - src.models.custom - INFO - Data shape: (188318, 130)
2025-12-21 20:08:43,387 - optuna.storages._in_memory - INFO - A new study created in memory with name: no-name-d7cc2911-cc0b-492e-a4fe-78cab26a5bb0


  0%|          | 0/5 [00:00<?, ?it/s]

2025-12-21 20:08:43,431 - src.models.custom - INFO - Initialized CustomPipeline with model_type=lightgbm, n_folds=5
2025-12-21 20:08:43,432 - src.models.custom - INFO - Starting 5-fold CV with lightgbm
2025-12-21 20:08:43,433 - src.models.custom - INFO - Data shape: (188318, 130), Cat cols: 116, Cont cols: 14
2025-12-21 20:08:43,435 - src.models.custom - INFO - Target stats: mean=3037.34, std=2904.08, min=0.67, max=121012.25
2025-12-21 20:08:43,438 - src.models.custom - INFO - Applied log1p transform to target. New stats: mean=7.6859, std=0.8113
2025-12-21 20:08:43,516 - src.models.custom - INFO - Fold 1/5: Train=150,654, Val=37,664
2025-12-21 20:15:39,025 - src.models.custom - INFO - Fold 1/5: Log MAE = 0.4118, Raw MAE = 1129.64 (took 415.6s)
2025-12-21 20:15:39,086 - src.models.custom - INFO - Fold 2/5: Train=150,654, Val=37,664
2025-12-21 20:22:33,400 - src.models.custom - INFO - Fold 2/5: Log MAE = 0.4144, Raw MAE = 1147.85 (took 414.4s)
2025-12-21 20:22:33,462 - src.models.custom 

In [9]:
tuned_lgb = tuner.get_best_pipeline()
tuned_lgb.fit(X, y, cat_columns, cont_columns)
log_mae_tuned, std_tuned = tuned_lgb.get_cv_score(metric="log_mae")
raw_mae_tuned, _ = tuned_lgb.get_cv_score(metric="raw_mae")

print(f"Tuned LightGBM CV Results:")
print(f"  Log MAE: {log_mae_tuned:.4f} (+/- {std_tuned:.4f})")
print(f"  Raw MAE: {raw_mae_tuned:.4f}")

tuned_lgb.save_model(models_dir / "tuned_lgb_pipeline.joblib")

2025-12-21 23:33:38,801 - src.models.custom - INFO - Initialized CustomPipeline with model_type=lightgbm, n_folds=5
2025-12-21 23:33:38,801 - src.models.custom - INFO - Starting 5-fold CV with lightgbm
2025-12-21 23:33:38,802 - src.models.custom - INFO - Data shape: (188318, 130), Cat cols: 116, Cont cols: 14
2025-12-21 23:33:38,803 - src.models.custom - INFO - Target stats: mean=3037.34, std=2904.08, min=0.67, max=121012.25
2025-12-21 23:33:38,805 - src.models.custom - INFO - Applied log1p transform to target. New stats: mean=7.6859, std=0.8113
2025-12-21 23:33:38,858 - src.models.custom - INFO - Fold 1/5: Train=150,654, Val=37,664
2025-12-21 23:40:33,583 - src.models.custom - INFO - Fold 1/5: Log MAE = 0.4118, Raw MAE = 1129.64 (took 414.8s)
2025-12-21 23:40:33,646 - src.models.custom - INFO - Fold 2/5: Train=150,654, Val=37,664
2025-12-21 23:47:27,512 - src.models.custom - INFO - Fold 2/5: Log MAE = 0.4144, Raw MAE = 1147.85 (took 413.9s)
2025-12-21 23:47:27,574 - src.models.custom 

## Взвешенный ансамбль

Объединение предсказаний моделей с весами, обратно пропорциональными их ошибкам:

**Формула весов:**
$$w_i = \frac{1/\text{LogMAE}_i}{\sum_j 1/\text{LogMAE}_j}$$


In [10]:
oof_lgb = lgb_pipeline.get_oof_predictions()
oof_xgb = xgb_pipeline.get_oof_predictions()
oof_catboost = catboost_pipeline.get_oof_predictions()

log_maes = np.array([log_mae_lgb, log_mae_xgb, log_mae_catboost])
weights = 1 / log_maes
weights = weights / weights.sum()

print(f"Веса ансамбля:")
print(f"  LightGBM: {weights[0]:.3f}")
print(f"  XGBoost: {weights[1]:.3f}")
print(f"  CatBoost: {weights[2]:.3f}")

oof_ensemble = weights[0] * oof_lgb + weights[1] * oof_xgb + weights[2] * oof_catboost
ensemble_metrics = compute_metrics(y.values, oof_ensemble)

print(f"\nEnsemble OOF Results:")
print(f"  Log MAE: {ensemble_metrics['log_mae']:.4f}")
print(f"  Raw MAE: {ensemble_metrics['mae']:.4f}")

Веса ансамбля:
  LightGBM: 0.335
  XGBoost: 0.334
  CatBoost: 0.331

Ensemble OOF Results:
  Log MAE: 0.4155
  Raw MAE: 1143.0141


## Сравнение результатов

Сравним метрики качества всех моделей на кросс-валидации.


In [11]:
results = pd.DataFrame({
    'Model': ['LAMA Baseline', 'LightGBM', 'XGBoost', 'CatBoost', 'Tuned LightGBM', 'Ensemble'],
    'Log MAE': [LAMA_BASELINE_LOG_MAE, log_mae_lgb, log_mae_xgb, log_mae_catboost, log_mae_tuned, ensemble_metrics['log_mae']],
    'Raw MAE': [LAMA_BASELINE_RAW_MAE, raw_mae_lgb, raw_mae_xgb, raw_mae_catboost, raw_mae_tuned, ensemble_metrics['mae']],
})
results = results.sort_values('Log MAE')

results.to_csv("custom_results.csv")

print(results.to_string(index=False))

best_custom = results[results['Model'] != 'LAMA Baseline'].iloc[0]
print(f"\nЛучшая модель: {best_custom['Model']}")
print(f"Log MAE: {best_custom['Log MAE']:.4f}")

         Model  Log MAE     Raw MAE
Tuned LightGBM 0.413043 1137.823730
      LightGBM 0.415481 1144.939626
      Ensemble 0.415489 1143.014123
       XGBoost 0.415809 1144.424819
      CatBoost 0.419900 1156.022977
 LAMA Baseline 0.436300 1187.000000

Лучшая модель: Tuned LightGBM
Log MAE: 0.4130


## Выводы и анализ результатов

### Результаты экспериментов

| Модель | Log MAE |
|--------|---------|
| LAMA Baseline | 0.4363 |
| LightGBM | 0.4155 |
| XGBoost | 0.4158 |
| CatBoost | 0.4199 | 
| Tuned LightGBM | **0.4130** | 
| Ensemble | 0.4155 |

### Ключевые наблюдения

1. **Лучший результат** — Tuned LightGBM
2. LightGBM, XGBoost, CatBoost показывают схожие результаты
3. **Optuna-тюнинг** дал дополнительное улучшение